In [1]:
import os
import json
from tqdm import tqdm
from datasets import Dataset, Features, Value # Value 用于定义特征类型
from typing import List, Dict, Any, Optional

def load_and_merge_json_files_to_hf_dataset(
    directory_path: str,
    features: Optional[Features] = None,
    encoding: str = 'utf-8'
) -> Optional[Dataset]:

    all_data_records: List[Dict[str, Any]] = []

    if not os.path.isdir(directory_path):
        print(f"Error: Directory '{directory_path}' not found.")
        return None

    print(f"Scanning directory: {directory_path}")
    for filename in tqdm(os.listdir(directory_path)):
        if filename.endswith(".json"):
            filepath = os.path.join(directory_path, filename)
            try:
                with open(filepath, 'r', encoding=encoding) as f:
                    # Check if file is empty first
                    content = f.read()
                    if not content.strip():
                        print(f"Skipping empty JSON file: {filepath}")
                        continue

                    # Reset file pointer to read JSON
                    f.seek(0)
                    data = json.load(f)

                if isinstance(data, dict):
                    all_data_records.append(data)
                elif isinstance(data, list):
                    # Ensure all items in the list are dictionaries
                    if all(isinstance(item, dict) for item in data):
                        all_data_records.extend(data)
                    else:
                        print(f"Warning: JSON file '{filepath}' is a list but contains non-dict items. Skipping.")
                else:
                    print(f"Warning: JSON file '{filepath}' does not contain a dict or list of dicts. Skipping.")

            except json.JSONDecodeError:
                print(f"Warning: Could not decode JSON from file: {filepath}. Skipping.")
            except Exception as e:
                print(f"Error reading file {filepath}: {e}. Skipping.")

    if not all_data_records:
        print("No valid data records found in JSON files to create a dataset.")
        return None

    print(f"Successfully loaded {len(all_data_records)} records from JSON files.")

    # Attempt to create Hugging Face Dataset
    try:
        if features:
            # If features are provided, use them
            hf_dataset = Dataset.from_list(all_data_records, features=features)
        else:
            # Try to infer features. This might not always work perfectly,
            # especially with complex or inconsistent data.
            print("No features provided, attempting to infer schema from data...")
            # For robust inference, it's good to ensure all dicts have the same keys,
            # but Dataset.from_list will try its best.
            hf_dataset = Dataset.from_list(all_data_records)
            print(f"Inferred features: {hf_dataset.features}")

        print(f"Hugging Face Dataset created successfully with {len(hf_dataset)} rows.")
        return hf_dataset

    except Exception as e:
        print(f"Error creating Hugging Face Dataset: {e}")
        return None

/cpfs02/user/liurunze/miniforge3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dd = load_and_merge_json_files_to_hf_dataset('/mnt/workspace/_/simpleRL-reason/_outputs/processbench/7B_ppo_reward')
dd

Scanning directory: /mnt/workspace/_/simpleRL-reason/_outputs/processbench/7B_ppo_reward


100%|██████████| 3400/3400 [00:10<00:00, 334.26it/s]


Successfully loaded 3400 records from JSON files.
No features provided, attempting to infer schema from data...
Inferred features: {'id': Value(dtype='string', id=None), 'generator': Value(dtype='string', id=None), 'problem': Value(dtype='string', id=None), 'steps': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'final_answer_correct': Value(dtype='bool', id=None), 'label': Value(dtype='int64', id=None), 'values': Sequence(feature=Sequence(feature=Value(dtype='float64', id=None), length=-1, id=None), length=-1, id=None), 'tag_indices': Sequence(feature=Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None), length=-1, id=None)}
Hugging Face Dataset created successfully with 3400 rows.


Dataset({
    features: ['id', 'generator', 'problem', 'steps', 'final_answer_correct', 'label', 'values', 'tag_indices'],
    num_rows: 3400
})

In [3]:
# # 前缀平均
# def process(data):
#     data['prm_values'] = []
#     for index in data['tag_indices'][0]:
#         llist = data['values'][0][:index]
#         v = sum(llist)/len(llist)
#         data['prm_values'].append(v)
    
#     return data

# 差值平均
def process(data):
    data['prm_values'] = []
    last_index = 0
    for index in data['tag_indices'][0]:
        llist = data['values'][0][last_index:index]
        v = sum(llist)/len(llist)
        data['prm_values'].append(v)
        last_index = index
    
    return data

# # 单位置值
# def process(data):
#     data['prm_values'] = []
#     for index in data['tag_indices'][0]:
#         v = data['values'][0][index-1]
#         data['prm_values'].append(v)
    
#     return data

dd_p = dd.map(process).remove_columns(['values', 'tag_indices'])
dd_p

Map: 100%|██████████| 3400/3400 [00:04<00:00, 760.16 examples/s]


Dataset({
    features: ['id', 'generator', 'problem', 'steps', 'final_answer_correct', 'label', 'prm_values'],
    num_rows: 3400
})

In [11]:
import numpy as np
from tqdm import tqdm

all_values_flat = []
for sample_values in tqdm(dd['values']):
    if sample_values:  # 确保内部列表不为空
        all_values_flat.extend(sample_values[0])

if not all_values_flat:
    print("数据集中没有 'values' 数据，或者所有 'values' 列表都为空。")
else:
    # 计算平均值
    average_value = np.mean(all_values_flat)

    # 计算最大值
    max_value = np.max(all_values_flat)

    # 计算最小值
    min_value = np.min(all_values_flat)

    print(f"所有 values 的平均值: {average_value}")
    print(f"所有 values 的最大值: {max_value}")
    print(f"所有 values 的最小值: {min_value}")

# 如果你的 'values' 列中的每个元素已经是 NumPy 数组或者可以轻松转换为 NumPy 数组
# 并且你想避免显式循环来展平列表（对于非常大的数据集可能更高效），可以这样做：

# 确保 'values' 列中的每个元素都是列表或可迭代对象
# 如果 'values' 本身是一个大的嵌套列表，可以先将其转换为NumPy数组（如果适用）
# 但对于 Dataset 对象，我们通常逐个处理行

# 另一种方法，如果 'values' 已经是数值类型且允许空列表
all_values_flat_np = []
for sublist in tqdm(dd['values']):
    if sublist: # 检查子列表是否为空
        all_values_flat_np.extend(sublist)


if all_values_flat_np:
    all_values_array = np.array(all_values_flat_np)
    print("\n--- 使用 NumPy 直接计算 ---")
    print(f"所有 values 的平均值: {np.mean(all_values_array)}")
    print(f"所有 values 的最大值: {np.max(all_values_array)}")
    print(f"所有 values 的最小值: {np.min(all_values_array)}")
else:
    print("\n--- 使用 NumPy 直接计算 ---")
    print("数据集中没有 'values' 数据，或者所有 'values' 列表都为空。")

100%|██████████| 3400/3400 [00:00<00:00, 45179.63it/s]


所有 values 的平均值: 1.073188616776399
所有 values 的最大值: 6.2734375
所有 values 的最小值: -11.078125


100%|██████████| 3400/3400 [00:00<00:00, 1860244.40it/s]


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (3400,) + inhomogeneous part.

In [40]:
dd_p[6]

{'id': 'math-925',
 'generator': 'Qwen2-7B-Instruct',
 'problem': 'On the number line shown, $AE$ = 40 cm, $AD$ = 30 cm, $BE$ = 20 cm, and $C$ is the midpoint of $\\overline{BD}$. In centimeters, what is $AC$? [asy] size(8cm);\npair A,B,C,D,E;\nA = (0,0);\nB = (2,0);\nD = (3,0);\nE = (4,0);\nC = (B+D)/2;\ndraw((-.5,0)--(4.5,0),Arrows);\ndot(A);\ndot(B);\ndot(C);\ndot(D);\ndot(E);\nlabel("$A$",A,S);\nlabel("$B$",B,S);\nlabel("$C$",C,S);\nlabel("$D$",D,S);\nlabel("$E$",E,S);\n[/asy]',
 'steps': ['To find the length of \\(AC\\), we can break down the given information and use it to form segments on the number line. First, we have: \\(AE = 40\\) cm, \\(AD = 30\\) cm, and \\(BE = 20\\) cm. Given that \\(C\\) is the midpoint of \\(\\overline{BD}\\), this implies that \\(BC = CD = \\frac{BD}{2}\\).',
  'Since \\(BE = 20\\) cm and \\(C\\) is the midpoint of \\(BD\\), we can infer that \\(BC = 10\\) cm (because \\(BE\\) is part of \\(BD\\) and \\(C\\) splits \\(BD\\) into two equal parts). Give